In [ ]:
# Setup: clone the StarX repo and the pinned TripoSR commit, install this
# notebook's dependencies, mount Drive, and report what machine we are on.
import os
import subprocess
import sys

BRANCH = "main"
TRIPOSR_COMMIT = "107cefdc244c39106fa830359024f6a2f1c78871"
NOTEBOOK_ID = "02"

IN_COLAB = os.path.exists("/content")
if IN_COLAB:
    REPO_DIR, TRIPOSR_DIR = "/content/StarX", "/content/TripoSR"
    if not os.path.exists(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--branch", BRANCH,
             "https://github.com/SattamAltwaim/StarX.git", REPO_DIR],
            check=True,
        )
else:
    REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
    TRIPOSR_DIR = os.path.join(REPO_DIR, "third_party", "TripoSR")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

if not os.path.exists(TRIPOSR_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/VAST-AI-Research/TripoSR.git",
         TRIPOSR_DIR],
        check=True,
    )
subprocess.run(["git", "-C", TRIPOSR_DIR, "checkout", "-q", TRIPOSR_COMMIT], check=True)

from starx import pins

assert pins.TRIPOSR_COMMIT == TRIPOSR_COMMIT, "notebook pin out of sync with starx/pins.py"
if pins.PIP_PINS[NOTEBOOK_ID]:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *pins.PIP_PINS[NOTEBOOK_ID]],
        check=True,
    )

from starx import colab as scolab

DRIVE = scolab.mount_drive()
report = scolab.setup_report()

# 02 - Sketches into pixels

The model cannot read parametric curves, so every design's sketches become a stack of grayscale images: one channel per timeline sketch, dark strokes on a mid-gray background, padded with blank channels up to a fixed count.

This notebook develops and validates that rasterizer on a single reference design, then batch-tests it on a random slice of the dataset. It writes nothing bulky - the real dataset build happens in notebook 03.

Any runtime works; no GPU is needed.

In [ ]:
# Configuration - every tunable for this notebook lives here.
import dataclasses
import json
import random
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from starx import fusion, rasterize, viz
from starx.config import StarXConfig

SMOKE = False   # True: smaller batch test for a fast pipeline check
SAMPLE_N = 60 if SMOKE else 300   # designs in the batch test
SEED = 1337

cfg = StarXConfig(
    drive_root=str(DRIVE / "StarX")
    if DRIVE is not None
    else os.path.join(REPO_DIR, "data", "StarX"),
    sketch_size=512,
    supersample=2,
    stroke_width=5,
    stroke_value=0,
    bg_value=128,
    margin=0.05,
    normalization_mode="shared",
    max_sketch_channels=6,
    include_construction=False,
    truncate_extra_sketches=True,
)
print("channels:", cfg.max_sketch_channels, " image:", cfg.sketch_size)

In [ ]:
# Load the reference design that ships with the repo as a test fixture.
FIXTURE_PATH = Path(REPO_DIR) / "tests" / "fixtures" / "20203_7e31e92a_0000.json"
design = fusion.load_design(FIXTURE_PATH)
for s in design.sketches:
    print(f"{s.name}: {len(s.curves)} curves, timeline index {s.timeline_index}")
print(f"extrudes: {design.n_extrudes}")
print(f"parser warnings: {design.warnings or 'none'}")

## From parametric curves to polylines

A sketch stores geometry as parametric curves, but drawing wants points. Each curve type gets sampled into a polyline:

- lines are just their two endpoints,
- circles and arcs are sampled analytically around their center (arcs pick the sweep orientation that lands on their stored endpoints),
- fitted splines are interpolated through their fit points, falling back to NURBS evaluation when only control points exist.

The result is a list of (N, 2) point arrays per sketch, in centimeters, ready to draw.

In [ ]:
# Sample every curve of the first sketch and draw the polylines, colored
# by curve type.
sketch = design.sketches[0]
palette = plt.get_cmap("tab10").colors
type_colors, handles = {}, {}
skipped_construction = 0
fig, ax = plt.subplots(figsize=(5.6, 5.6))
for curve in sketch.curves.values():
    if curve.get("construction_geom") and not cfg.include_construction:
        skipped_construction += 1
        continue
    poly = fusion.sample_curve(curve, sketch.points, cfg.include_construction)
    if poly is None:
        continue
    ctype = curve["type"]
    color = type_colors.setdefault(ctype, palette[len(type_colors)])
    (line,) = ax.plot(poly[:, 0], poly[:, 1], color=color, linewidth=2)
    handles[ctype] = line
ax.set_aspect("equal")
ax.legend(handles.values(), handles.keys())
ax.set_title(f"{sketch.name} - curves as polylines (cm)")
plt.show()
print(f"construction curves skipped: {skipped_construction}")

In [ ]:
# Profiles are the closed regions that extrudes turn into solid material.
# Draw their outlines (straight-segment view) over the sketch curves.
fig, ax = plt.subplots(figsize=(5.6, 5.6))
for curve in sketch.curves.values():
    poly = fusion.sample_curve(curve, sketch.points)
    if poly is not None:
        ax.plot(poly[:, 0], poly[:, 1], color="0.82", linewidth=1.2)
palette = plt.get_cmap("tab10").colors
for i, profile in enumerate(sketch.profiles.values()):
    for loop in profile.get("loops", []):
        for segment in loop.get("profile_curves", []):
            s, e = segment["start_point"], segment["end_point"]
            ax.plot(
                [s["x"], e["x"]], [s["y"], e["y"]],
                color=palette[i % 10], linewidth=2.4,
            )
ax.set_aspect("equal")
ax.set_title(f"{sketch.name} - profile outlines over curves")
plt.show()
print(f"profiles in {sketch.name}: {len(sketch.profiles)}")

## One scale for the whole design

Each channel must frame its sketch inside the image, which forces a choice:

- **shared** (the default): one pixels-per-centimeter factor for the whole design, chosen so the largest sketch fills the frame minus a margin. A small locating hole stays small next to the big base plate - relative size is real geometric information the model can use.
- **per_sketch**: every sketch is scaled to fill its own frame. Maximum legibility per channel, but all size relationships between sketches are destroyed.

Every channel is centered on its own sketch either way. The comparison below shows what the trade-off looks like in practice.

In [ ]:
# Rasterize the same design under both modes and compare the strips.
strips = {}
for mode in ("shared", "per_sketch"):
    mode_cfg = dataclasses.replace(cfg, normalization_mode=mode)
    stack_m, meta_m = rasterize.rasterize_design(design, mode_cfg)
    strips[mode] = (rasterize.stack_to_strip(stack_m), meta_m)

fig, axes = plt.subplots(2, 1, figsize=(2.1 * cfg.max_sketch_channels, 4.8))
for ax, (mode, (strip_m, _)) in zip(axes, strips.items()):
    ax.imshow(strip_m, cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"{mode} normalization", fontsize=10)
    ax.axis("off")
plt.show()
for mode, (_, meta_m) in strips.items():
    print(f"{mode:>10} px_per_cm: {meta_m['px_per_cm']}")

In [ ]:
# The full channel stack for this design: one channel per timeline sketch,
# blank padding after the last one.
stack, meta = rasterize.rasterize_design(design, cfg)
fig = viz.show_sketch_stack(stack, meta, title=design.design_id)
plt.show()
print({k: meta[k] for k in ("n_sketches_total", "truncated", "blank_channels")})

In [ ]:
# Sanity: the raster channel should match the vector drawing geometrically.
# Both panels show +y pointing up, so shapes must agree by eye.
fig, axes = plt.subplots(1, 2, figsize=(9.4, 4.7))
axes[0].imshow(stack[0], cmap="gray", vmin=0, vmax=255)
axes[0].set_title("rasterized channel 0")
axes[0].axis("off")
first_sketch = design.sketches[0]
for curve in first_sketch.curves.values():
    poly = fusion.sample_curve(curve, first_sketch.points)
    if poly is not None:
        axes[1].plot(poly[:, 0], poly[:, 1], color="0.1", linewidth=2)
axes[1].set_aspect("equal")
axes[1].set_title("vector curves (cm)")
plt.show()

In [ ]:
# Batch-test the rasterizer on random designs straight from the zip.
# Keep only metadata (stacks are large); save a few stacks for the gallery.
zip_local_dir = "/content" if IN_COLAB else os.path.join(REPO_DIR, "data")
zip_path = scolab.ensure_zip_local(cfg, local_dir=zip_local_dir)
names = scolab.zip_inventory(zip_path)
json_members = [
    n
    for n in names
    if n.endswith(".json") and "train_test" not in os.path.basename(n).lower()
]
design_ids = sorted(os.path.basename(n)[:-5] for n in json_members)

rng = random.Random(SEED)
batch_ids = rng.sample(design_ids, min(SAMPLE_N, len(design_ids)))
metas, failures, gallery = [], [], []
with zipfile.ZipFile(zip_path) as zf:
    for design_id in tqdm(batch_ids, desc="rasterizing"):
        try:
            d = fusion.load_design(
                json.load(zf.open(scolab.json_member(names, design_id)))
            )
            stack_i, meta_i = rasterize.rasterize_design(d, cfg)
        except Exception as error:
            failures.append({"design_id": design_id, "error": repr(error)})
            continue
        metas.append(meta_i)
        if len(gallery) < 8 and not meta_i["all_blank"]:
            gallery.append((design_id, stack_i, meta_i))

print(f"clean rasterization: {len(metas) / len(batch_ids):.1%}")
pd.DataFrame(failures).head(10) if failures else print("no failures")

In [ ]:
# A gallery of channel strips from the batch - the model's-eye view of
# several different designs.
fig, axes = plt.subplots(
    len(gallery), 1, figsize=(2.0 * cfg.max_sketch_channels, 2.15 * len(gallery))
)
for ax, (design_id, stack_g, meta_g) in zip(np.atleast_1d(axes), gallery):
    ax.imshow(rasterize.stack_to_strip(stack_g), cmap="gray", vmin=0, vmax=255)
    ax.set_title(
        f"{design_id}  ({meta_g['n_sketches_total']} sketches)", fontsize=9
    )
    ax.axis("off")
fig.tight_layout()
plt.show()

In [ ]:
# Edge cases: the busiest design in the batch (shows truncation at the
# channel cap) and how often a design has nothing drawable at all.
meta_df = pd.DataFrame(metas)
busiest_id = meta_df.loc[meta_df["n_sketches_total"].idxmax(), "design_id"]
with zipfile.ZipFile(zip_path) as zf:
    busiest = fusion.load_design(
        json.load(zf.open(scolab.json_member(names, busiest_id)))
    )
stack_b, meta_b = rasterize.rasterize_design(busiest, cfg)
fig = viz.show_sketch_stack(stack_b, meta_b, title=f"{busiest_id} - truncation view")
plt.show()
print(
    f"total sketches {meta_b['n_sketches_total']}, "
    f"kept {meta_b['n_channels_used']}, truncated: {meta_b['truncated']}"
)
print(f"designs truncated in batch: {meta_df['truncated'].mean():.1%}")
print(
    f"designs with nothing drawable: "
    f"{int(meta_df['all_blank'].sum())} of {len(meta_df)}"
)

In [ ]:
# Styling ablation, visual only - the configured default stays. The gray
# background is deliberate: it matches how TripoSR saw objects in
# pretraining (foreground composited over mid-gray).
variants = [
    ("thin stroke", dict(stroke_width=3)),
    ("default", {}),
    ("thick stroke", dict(stroke_width=9)),
    ("white background", dict(bg_value=255)),
]
fig, axes = plt.subplots(1, len(variants), figsize=(3.1 * len(variants), 3.4))
for ax, (label, overrides) in zip(axes, variants):
    stack_v, _ = rasterize.rasterize_design(
        design, dataclasses.replace(cfg, **overrides)
    )
    ax.imshow(stack_v[0], cmap="gray", vmin=0, vmax=255)
    ax.set_title(label, fontsize=10)
    ax.axis("off")
plt.show()

In [ ]:
# Shards will store each stack as one horizontal strip PNG - confirm the
# encode/decode roundtrip is bit-exact and look at the stored form.
strip = rasterize.stack_to_strip(stack)
back = rasterize.strip_to_stack(strip, cfg.max_sketch_channels)
assert np.array_equal(back, stack), "strip roundtrip must be bit-exact"

fig, ax = plt.subplots(figsize=(2.0 * cfg.max_sketch_channels, 2.4))
ax.imshow(strip, cmap="gray", vmin=0, vmax=255)
ax.axis("off")
ax.set_title("the strip exactly as it will be stored in shards")
plt.show()
print("roundtrip exact:", strip.shape, "->", back.shape)

## Takeaways

The rasterizer is validated: every frequent curve type draws correctly, the channel stack matches the vector geometry, padding and truncation behave as configured, and the strip storage format is bit-exact.

Notebook 03 runs this at dataset scale and adds the other half of each training sample: ground-truth renders of the design's final mesh.